<a href="https://colab.research.google.com/github/jose02082000/Curso-programacion-cientifica/blob/main/ejercicios/taller-evaluativo-02-funciones-y-poo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Ejercicio 1 — Calibrar una lectura
En este ejercicio definimos la función `calibrar_lectura` que aplica una recta de calibración ($y = m \cdot x + b$) a un valor dado. Incluye parámetros opcionales con valores por defecto que mantienen el valor inalterado si no se especifican.

In [3]:
def calibrar_lectura(valor, pendiente=1.0, intercepto=0.0):
    """Aplica una recta de calibracion a un valor de lectura."""
    return valor * pendiente + intercepto

sin_calibrar = calibrar_lectura(10.0)
calibrada = calibrar_lectura(10.0, 1.2, -0.5)
con_intercepto = calibrar_lectura(10.0, intercepto=2.0)

print(f"Sin calibrar: {sin_calibrar}")
print(f"Calibrada: {calibrada}")
print(f"Con intercepto: {con_intercepto}")

Sin calibrar: 10.0
Calibrada: 11.5
Con intercepto: 12.0


## Ejercicio 2 — ¿Es utilizable la lectura?
Definimos la función `es_lectura_valida` para comprobar si un dato es útil. Retorna `True` si el valor no coincide con el centinela (-999.0) y se encuentra dentro del rango físico del sensor [0.0, 500.0].

In [4]:
lecturas_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]

def es_lectura_valida(valor):
    """Determina si una lectura esta dentro del rango valido y no es un centinela."""
    return valor != -999.0 and 0.0 <= valor <= 500.0

print(f"Lectura 25.0 valida: {es_lectura_valida(25.0)}")
print(f"Lectura -999.0 valida: {es_lectura_valida(-999.0)}")
print(f"Lectura 600.0 valida: {es_lectura_valida(600.0)}")
print(f"Lectura 0.0 valida: {es_lectura_valida(0.0)}")

Lectura 25.0 valida: True
Lectura -999.0 valida: False
Lectura 600.0 valida: False
Lectura 0.0 valida: True


## Ejercicio 3 — Promedio de las lecturas útiles
Creamos la función `promedio_valido` que calcula el promedio únicamente de las lecturas útiles dentro de una lista. Hace uso directo de `es_lectura_valida` y devuelve `None` en caso de que no existan lecturas válidas.

In [5]:
def promedio_valido(lecturas):
    """Calcula el promedio de las lecturas validas de una lista."""
    suma = 0.0
    contador = 0
    for x in lecturas:
        if es_lectura_valida(x):
            suma += x
            contador += 1
    return suma / contador if contador > 0 else None

promedio_sbc_a = promedio_valido(lecturas_sbc_a)

print(f"Promedio SBC-A: {promedio_sbc_a}")
print(f"Promedio lista no valida: {promedio_valido([-999.0, -999.0])}")

Promedio SBC-A: 27.0
Promedio lista no valida: None


## Ejercicio 4 — Calibrar la serie con map, filter y lambda
Utilizamos programación funcional con `filter` y `map` para procesar la serie `lecturas_sbc_a`. Se filtran las lecturas inválidas, se calibran los datos con la pendiente e intercepto indicados y se contabilizan las excedencias a la categoría Moderada (> 35.4 µg/m³).

In [6]:
pendiente_sbc_a = 1.2
intercepto_sbc_a = -0.5

utiles = filter(es_lectura_valida, lecturas_sbc_a)
lecturas_calibradas = list(map(lambda x: calibrar_lectura(x, pendiente_sbc_a, intercepto_sbc_a), utiles))
excedencias = len(list(filter(lambda x: x > 35.4, lecturas_calibradas)))

print(f"Lecturas calibradas: {lecturas_calibradas}")
print(f"Cantidad de excedencias: {excedencias}")

Lecturas calibradas: [11.5, 29.5, 47.5, 9.1, 61.9]
Cantidad de excedencias: 2


## Ejercicio 5 — La clase Lectura
Implementamos la clase `Lectura` que representa una medición puntual. Incluye métodos para convertir unidades a microgramos por metro cúbico (`a_microgramos`) y detectar datos faltantes (`es_dato_faltante`).

In [7]:
class Lectura:
    """Representa una lectura de calidad del aire con su unidad y sensor."""
    def __init__(self, valor, unidad="ug/m3", codigo_sensor="sin asignar"):
        """Inicializa los atributos de la lectura."""
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Convierte el valor a ug/m3 si la unidad esta en mg/m3."""
        return self.valor * 1000.0 if self.unidad == "mg/m3" else self.valor

    def es_dato_faltante(self):
        """Indica si el valor corresponde al centinela de dato faltante."""
        return self.valor == -999.0

lectura_campo = Lectura(0.0252, "mg/m3", "SBC-A")
lectura_faltante = Lectura(-999.0, codigo_sensor="SBC-A")
lectura_normal = Lectura(30.0)

print(f"Lectura campo en ug/m3: {lectura_campo.a_microgramos()}")
print(f"Lectura faltante es dato faltante: {lectura_faltante.es_dato_faltante()}")
print(f"Lectura campo es dato faltante: {lectura_campo.es_dato_faltante()}")
print(f"Lectura normal -> unidad: {lectura_normal.unidad}, sensor: {lectura_normal.codigo_sensor}")

Lectura campo en ug/m3: 25.2
Lectura faltante es dato faltante: True
Lectura campo es dato faltante: False
Lectura normal -> unidad: ug/m3, sensor: sin asignar


## Ejercicio 6 — La clase Sensor: validación y encapsulamiento
Implementamos la clase `Sensor`, que encapsula sus atributos de calibración (`_pendiente` e `_intercepto`) mediante atributos internos protegidos y realiza la validación de parámetros al construirse.

In [8]:
class Sensor:
    """Representa un sensor de calibracion con atributos protegidos."""
    def __init__(self, codigo, pendiente=1.0, intercepto=0.0):
        """Valida y asigna las propiedades del sensor."""
        if not codigo:
            raise ValueError("El codigo no puede ser un texto vacio.")
        if pendiente <= 0:
            raise ValueError("La pendiente debe ser un valor positivo.")
        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, valor):
        """Aplica la funcion de calibracion usando las propiedades del sensor."""
        return calibrar_lectura(valor, self._pendiente, self._intercepto)

    def __repr__(self):
        """Devuelve la representacion formal del objeto Sensor."""
        return f"Sensor(codigo='{self.codigo}', pendiente={self._pendiente}, intercepto={self._intercepto})"

    def __str__(self):
        """Devuelve una representacion legible del objeto Sensor."""
        return f"Sensor {self.codigo} (Pendiente: {self._pendiente}, Intercepto: {self._intercepto})"

sensor_a = Sensor("SBC-A", 1.2, -0.5)
sensor_b = Sensor("SBC-B")

print(f"sensor_a (__str__): {sensor_a}")
print(f"Lista de sensores (__repr__): {[sensor_a, sensor_b]}")
print(f"Calibracion 10.0 con sensor_a: {sensor_a.calibrar(10.0)}")
print(f"Calibracion 10.0 con sensor_b: {sensor_b.calibrar(10.0)}")

sensor_a (__str__): Sensor SBC-A (Pendiente: 1.2, Intercepto: -0.5)
Lista de sensores (__repr__): [Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5), Sensor(codigo='SBC-B', pendiente=1.0, intercepto=0.0)]
Calibracion 10.0 con sensor_a: 11.5
Calibracion 10.0 con sensor_b: 10.0


## Ejercicio 7 — La clase Estacion: objetos que contienen objetos
Construimos la clase `Estacion` para agrupar múltiples objetos de tipo `Lectura`. El método `valores` extrae y convierte la concentración de cada medición en $\mu g/m^3$.

In [9]:
class Estacion:
    """Representa una estacion de monitoreo que almacena lecturas."""
    def __init__(self, codigo, sector):
        """Inicializa la estacion con codigo, sector y lista vacia de lecturas."""
        self.codigo = codigo
        self.sector = sector
        self._lecturas = []

    def agregar(self, lectura):
        """Agrega un objeto Lectura a la lista interna."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve las concentraciones en ug/m3 de todas las lecturas."""
        return [l.a_microgramos() for l in self._lecturas]

    def __repr__(self):
        """Devuelve la representacion formal de la Estacion."""
        return f"Estacion('{self.codigo}', '{self.sector}', lecturas={len(self._lecturas)})"

datos_campana = [
    (10.0, "ug/m3"), (-999.0, "ug/m3"), (0.025, "mg/m3"), (40.0, "ug/m3"),
    (-999.0, "ug/m3"), (8.0, "ug/m3"), (52.0, "ug/m3"), (600.0, "ug/m3"),
]

estacion_oriente = Estacion("EST-04", "Oriente")

for valor, unidad in datos_campana:
    estacion_oriente.agregar(Lectura(valor, unidad, "SBC-A"))

print(f"Representacion de la estacion: {estacion_oriente}")
print(f"Valores de la estacion en ug/m3: {estacion_oriente.valores()}")

Representacion de la estacion: Estacion('EST-04', 'Oriente', lecturas=8)
Valores de la estacion en ug/m3: [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


## Ejercicio 8 — Integración: el informe de la estación
Creamos la función `categoria_pm25` y la clase `InformeEstacion` para integrar el filtrado, calibración y clasificación de las mediciones de una estación asociadas a un sensor.

In [10]:
def categoria_pm25(concentracion):
    """Devuelve la categoria del ICA segun el nivel de concentracion de PM2.5."""
    if concentracion <= 12.0:
        return "Buena"
    elif concentracion <= 35.4:
        return "Moderada"
    elif concentracion <= 55.4:
        return "Dañina para grupos sensibles"
    else:
        return "Dañina para la salud"

class InformeEstacion:
    """Genera informes integrados a partir de una estacion y un sensor."""
    def __init__(self, estacion, sensor):
        """Inicializa el informe recibiendo objetos Estacion y Sensor."""
        self._estacion = estacion
        self._sensor = sensor

    def promedio_calibrado(self):
        """Calcula el promedio calibrado de las lecturas utiles de la estacion."""
        vals = self._estacion.valores()
        utiles = filter(es_lectura_valida, vals)
        calibrados = list(map(lambda x: self._sensor.calibrar(x), utiles))
        return promedio_valido(calibrados)

    def categoria(self):
        """Determina la categoria de calidad del aire del promedio calibrado."""
        return categoria_pm25(self.promedio_calibrado())

    def __str__(self):
        """Formatea el informe de acuerdo a la especificacion."""
        prom = round(self.promedio_calibrado(), 1)
        return f"{self._estacion.codigo}: {prom} ug/m3 - {self.categoria()}"

informe = InformeEstacion(estacion_oriente, sensor_a)

print(f"Informe formateado: {informe}")
print(f"Promedio calibrado exacto: {informe.promedio_calibrado()}")

Informe formateado: EST-04: 31.9 ug/m3 - Moderada
Promedio calibrado exacto: 31.9


## Verificación Final
Celda de comprobación de integridad del taller.

In [11]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"
assert str(informe) == "EST-04: 31.9 ug/m3 - Moderada", "revise __str__ de InformeEstacion"
print("Verificación completada sin errores.")

Verificación completada sin errores.
